# Sector Prompt Search

This notebook mirrors the `report_type_prompt_search` pipeline for the `sector` metadata task. It evaluates 15 prompts on a balanced sample across the 11 supported sectors.

In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
sector_verbolizer = {
    "Technology": ["technology", "tech", "software", "semiconductor"],
    "Healthcare": ["healthcare", "medical", "pharma", "biotech"],
    "Financial Services": ["financial", "banking", "finance", "fin"],
    "Consumer Cyclical": ["cyclical", "consumer", "retail", "auto"],
    "Consumer Defensive": ["defensive", "staples", "food", "beverage"],
    "Industrials": ["industrials", "industrial", "indust", "manufacturing"],
    "Energy": ["energy", "oil", "gas", "natural"],
    "Utilities": ["utilities", "utility", "util", "electric"],
    "Real Estate": ["real", "estate", "property", "properties", "reit",],
    "Basic Materials": ["materials", "material", "mining", "basic"],
    "Communication Services": ["entertainment", "media", "communications", "commun", "internet",],
}


sector_prompts = [
    "The sector of this company are:",
    "The sector are:",
    "I think this company belongs to a sector of:", # Best one 
    "My investment analyzis tell this company's sector is:",
    "Sector:",
    "Among different options: industrials, healthcare, energy, utilities, technology, materials, communication, cyclical, financials, defensive, realestate. Sector:",
    "What is the company sector? One-word label only from the allowed list.",
    "Sector prediction task. Return exactly one allowed sector token.",
    "Choose the issuer's primary business sector from this list: Technology, Healthcare, Financial Services, Consumer Cyclical, Consumer Defensive, Industrials, Energy, Utilities, Real Estate, Basic Materials, Communication Services. Sector:",
    "You are building a cross-sectional asset-pricing dataset. Tag this filing with one sector class only so it can be used as a fixed effect. Assigned sector:",
    "Ignore legal boilerplate and focus on the economic engine of the firm (what it sells, to whom, and in which market). The closest sector class is:",
    "Rapid-fire classification: if you had 2 seconds to bucket this company for portfolio construction, which sector would you pick? Answer:",
    "If evidence is mixed, choose the sector most supported by recurring business descriptions in the excerpt. Final sector:",
    "Assume you are an equity research associate preparing metadata for downstream return-prediction models. Your objective is to infer the company's dominant sector from the filing language, not to summarize the report. We care about the primary operating domain reflected by recurring terms, product categories, customer markets, and regulatory context. Use the 11-sector taxonomy (Technology, Healthcare, Financial Services, Consumer Cyclical, Consumer Defensive, Industrials, Energy, Utilities, Real Estate, Basic Materials, Communication Services), resolve ambiguity by choosing the most central business line, and provide one final sector label.",
    "You are performing high-precision metadata annotation for a research dataset linking textual filings to firm characteristics. The annotation rule is simple but strict: one filing, one sector label, chosen from 11 predefined classes. Determine which class best captures the issuer's core line of business by weighing explicit activity descriptors, product-market focus, and sector-specific vocabulary patterns. If multiple sectors appear, assign the sector that most likely drives long-run fundamentals and investor classification conventions. The permitted labels are Technology, Healthcare, Financial Services, Consumer Cyclical, Consumer Defensive, Industrials, Energy, Utilities, Real Estate, Basic Materials, and Communication Services. Output only the final selected sector label.",
]

In [3]:
model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},
)

model_driver = ModelDriver(model_name=model_name, model=model)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.14it/s]
2026-05-29 12:02:33,734 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [4]:
fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=["raw_text"])
reports_df = fetcher._apply_company_filters(reports_df, {})
reports_df = reports_df[reports_df["sector"].isin(sector_verbolizer.keys())].copy()
reports_df = reports_df[reports_df["raw_text"].notna()].copy()

sector_counts = reports_df["sector"].value_counts().sort_index()
n_per_sector = min(5, int(sector_counts.min()))
if n_per_sector < 1:
    raise ValueError("No samples available for at least one sector.")

sample_parts = []
for sector in sector_verbolizer:
    sector_df = reports_df[reports_df["sector"] == sector]
    sample_parts.append(sector_df.sample(n=n_per_sector, random_state=42))

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"Rows per sector in sample: {n_per_sector}")
display(sample_df["sector"].value_counts().sort_index())
sample_df[["id", "sector"]].head()

/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:130: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)
Rows per sector in sample: 5


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  companies_df = pd.read_sql_query(query, conn)


sector
Basic Materials           5
Communication Services    5
Consumer Cyclical         5
Consumer Defensive        5
Energy                    5
Financial Services        5
Healthcare                5
Industrials               5
Real Estate               5
Technology                5
Utilities                 5
Name: count, dtype: int64

,id,sector
0,3706,Energy
1,82404,Healthcare
2,36467,Energy
3,110801,Financial Services
4,4288,Consumer Cyclical


In [ ]:
def evaluate_prompt(prompt: str, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    rows = []
    for row in tqdm(data.itertuples(index=False), total=len(data), desc=prompt[:48]):
        result = model_driver.predict_metadata(
            verbolizer=sector_verbolizer,
            prompt=prompt,
            text=row.raw_text,
        )
        probabilities = result["probabilities"]
        predicted_label = max(probabilities, key=probabilities.get)
        rows.append(
            {
                "id": row.id,
                "true_label": row.sector,
                "predicted_label": predicted_label,
                "confidence": result["confidence"],
                "true_label_probability": probabilities[row.sector],
            }
        )

    predictions_df = pd.DataFrame(rows)
    y_true = predictions_df["true_label"]
    y_pred = predictions_df["predicted_label"]

    metrics = {
        "prompt": prompt,
        "avg_confidence": predictions_df["confidence"].mean(),
        "avg_true_label_probability": predictions_df["true_label_probability"].mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    return metrics, predictions_df


all_metrics = []
all_predictions = {}

for prompt in sector_prompts:
    metrics, predictions_df = evaluate_prompt(prompt, sample_df)
    all_metrics.append(metrics)
    all_predictions[prompt] = predictions_df

    print("\nPROMPT:", prompt)
    print(predictions_df["predicted_label"].value_counts())
    print(pd.crosstab(predictions_df["true_label"], predictions_df["predicted_label"]))

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values(["f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>prompt</th>
      <th>avg_confidence</th>
      <th>avg_true_label_probability</th>
      <th>accuracy</th>
      <th>precision_macro</th>
      <th>recall_macro</th>
      <th>f1_macro</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>I think this company belongs to a sector of:</td>
      <td>0.308681</td>
      <td>0.519298</td>
      <td>0.727273</td>
      <td>0.764683</td>
      <td>0.727273</td>
      <td>0.698167</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Ignore legal boilerplate and focus on the econ...</td>
      <td>0.235236</td>
      <td>0.509157</td>
      <td>0.709091</td>
      <td>0.703139</td>
      <td>0.709091</td>
      <td>0.681750</td>
    </tr>
    <tr>
      <th>2</th>
      <td>My investment analyzis tell this company's sec...</td>
      <td>0.284394</td>
      <td>0.519921</td>
      <td>0.727273</td>
      <td>0.671392</td>
      <td>0.727273</td>
      <td>0.678470</td>
    </tr>
    <tr>
      <th>3</th>
      <td>You are building a cross-sectional asset-prici...</td>
      <td>0.337304</td>
      <td>0.493033</td>
      <td>0.672727</td>
      <td>0.636580</td>
      <td>0.672727</td>
      <td>0.629117</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Among different options: industrials, healthca...</td>
      <td>0.282611</td>
      <td>0.387059</td>
      <td>0.618182</td>
      <td>0.713836</td>
      <td>0.618182</td>
      <td>0.613564</td>
    </tr>
    <tr>
      <th>5</th>
      <td>Rapid-fire classification: if you had 2 second...</td>
      <td>0.251523</td>
      <td>0.491360</td>
      <td>0.654545</td>
      <td>0.645896</td>
      <td>0.654545</td>
      <td>0.607186</td>
    </tr>
    <tr>
      <th>6</th>
      <td>Sector:</td>
      <td>0.228934</td>
      <td>0.485101</td>
      <td>0.600000</td>
      <td>0.667208</td>
      <td>0.600000</td>
      <td>0.594988</td>
    </tr>
    <tr>
      <th>7</th>
      <td>The sector of this company are:</td>
      <td>0.177956</td>
      <td>0.453000</td>
      <td>0.600000</td>
      <td>0.691531</td>
      <td>0.600000</td>
      <td>0.584514</td>
    </tr>
    <tr>
      <th>8</th>
      <td>Choose the issuer's primary business sector fr...</td>
      <td>0.463376</td>
      <td>0.433373</td>
      <td>0.545455</td>
      <td>0.631267</td>
      <td>0.545455</td>
      <td>0.538576</td>
    </tr>
    <tr>
      <th>9</th>
      <td>If evidence is mixed, choose the sector most s...</td>
      <td>0.178227</td>
      <td>0.302258</td>
      <td>0.436364</td>
      <td>0.472727</td>
      <td>0.436364</td>
      <td>0.407779</td>
    </tr>
    <tr>
      <th>10</th>
      <td>The sector are:</td>
      <td>0.053213</td>
      <td>0.275815</td>
      <td>0.290909</td>
      <td>0.408517</td>
      <td>0.290909</td>
      <td>0.294770</td>
    </tr>
    <tr>
      <th>11</th>
      <td>What is the company sector? One-word label onl...</td>
      <td>0.021106</td>
      <td>0.283674</td>
      <td>0.290909</td>
      <td>0.284848</td>
      <td>0.290909</td>
      <td>0.241919</td>
    </tr>
    <tr>
      <th>12</th>
      <td>Sector prediction task. Return exactly one all...</td>
      <td>0.002283</td>
      <td>0.156650</td>
      <td>0.200000</td>
      <td>0.318531</td>
      <td>0.200000</td>
      <td>0.177814</td>
    </tr>
    <tr>
      <th>13</th>
      <td>You are performing high-precision metadata ann...</td>
      <td>0.001933</td>
      <td>0.134112</td>
      <td>0.200000</td>
      <td>0.215111</td>
      <td>0.200000</td>
      <td>0.115906</td>
    </tr>
    <tr>
      <th>14</th>
      <td>Assume you are an equity research associate pr...</td>
      <td>0.002447</td>
      <td>0.125034</td>
      <td>0.163636</td>
      <td>0.147499</td>
      <td>0.163636</td>
      <td>0.099172</td>
    </tr>
  </tbody>
</table>
</div>

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>prompt</th>
      <th>avg_confidence</th>
      <th>avg_true_label_probability</th>
      <th>accuracy</th>
      <th>precision_macro</th>
      <th>recall_macro</th>
      <th>f1_macro</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>I think this company belongs to a sector of:</td>
      <td>0.289985</td>
      <td>0.500339</td>
      <td>0.690909</td>
      <td>0.749187</td>
      <td>0.690909</td>
      <td>0.664394</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Ignore legal boilerplate and focus on the econ...</td>
      <td>0.215730</td>
      <td>0.483392</td>
      <td>0.654545</td>
      <td>0.716595</td>
      <td>0.654545</td>
      <td>0.635314</td>
    </tr>
    <tr>
      <th>2</th>
      <td>My investment analyzis tell this company's sec...</td>
      <td>0.264048</td>
      <td>0.493283</td>
      <td>0.672727</td>
      <td>0.595455</td>
      <td>0.672727</td>
      <td>0.611511</td>
    </tr>
    <tr>
      <th>3</th>
      <td>The sector of this company are:</td>
      <td>0.170484</td>
      <td>0.448126</td>
      <td>0.600000</td>
      <td>0.716206</td>
      <td>0.600000</td>
      <td>0.586003</td>
    </tr>
    <tr>
      <th>4</th>
      <td>Among different options: industrials, healthca...</td>
      <td>0.269130</td>
      <td>0.371320</td>
      <td>0.563636</td>
      <td>0.635227</td>
      <td>0.563636</td>
      <td>0.543967</td>
    </tr>
    <tr>
      <th>5</th>
      <td>Rapid-fire classification: if you had 2 second...</td>
      <td>0.235929</td>
      <td>0.460932</td>
      <td>0.600000</td>
      <td>0.569487</td>
      <td>0.600000</td>
      <td>0.543792</td>
    </tr>
    <tr>
      <th>6</th>
      <td>You are building a cross-sectional asset-prici...</td>
      <td>0.323157</td>
      <td>0.463328</td>
      <td>0.600000</td>
      <td>0.542857</td>
      <td>0.600000</td>
      <td>0.538639</td>
    </tr>
    <tr>
      <th>7</th>
      <td>Sector:</td>
      <td>0.215881</td>
      <td>0.463915</td>
      <td>0.527273</td>
      <td>0.549784</td>
      <td>0.527273</td>
      <td>0.492182</td>
    </tr>
    <tr>
      <th>8</th>
      <td>Choose the issuer's primary business sector fr...</td>
      <td>0.431688</td>
      <td>0.403117</td>
      <td>0.490909</td>
      <td>0.559907</td>
      <td>0.490909</td>
      <td>0.473626</td>
    </tr>
    <tr>
      <th>9</th>
      <td>If evidence is mixed, choose the sector most s...</td>
      <td>0.167024</td>
      <td>0.284750</td>
      <td>0.400000</td>
      <td>0.424765</td>
      <td>0.400000</td>
      <td>0.363810</td>
    </tr>
    <tr>
      <th>10</th>
      <td>The sector are:</td>
      <td>0.051853</td>
      <td>0.274230</td>
      <td>0.309091</td>
      <td>0.417749</td>
      <td>0.309091</td>
      <td>0.307276</td>
    </tr>
    <tr>
      <th>11</th>
      <td>What is the company sector? One-word label onl...</td>
      <td>0.020521</td>
      <td>0.284598</td>
      <td>0.290909</td>
      <td>0.284091</td>
      <td>0.290909</td>
      <td>0.240796</td>
    </tr>
    <tr>
      <th>12</th>
      <td>Sector prediction task. Return exactly one all...</td>
      <td>0.002199</td>
      <td>0.155830</td>
      <td>0.163636</td>
      <td>0.181748</td>
      <td>0.163636</td>
      <td>0.120950</td>
    </tr>
    <tr>
      <th>13</th>
      <td>You are performing high-precision metadata ann...</td>
      <td>0.001924</td>
      <td>0.134046</td>
      <td>0.200000</td>
      <td>0.214452</td>
      <td>0.200000</td>
      <td>0.115010</td>
    </tr>
    <tr>
      <th>14</th>
      <td>Assume you are an equity research associate pr...</td>
      <td>0.002377</td>
      <td>0.125838</td>
      <td>0.145455</td>
      <td>0.055901</td>
      <td>0.145455</td>
      <td>0.068200</td>
    </tr>
  </tbody>
</table>
</div>

In [6]:
top_3_prompts = results_df.head(3).copy()
display(top_3_prompts[["prompt", "avg_confidence", "precision_macro", "recall_macro", "f1_macro", "accuracy"]])

for idx, row in top_3_prompts.iterrows():
    print(f"\nTop {idx + 1} prompt")
    print(row["prompt"])
    print(
        {
            "avg_confidence": row["avg_confidence"],
            "precision_macro": row["precision_macro"],
            "recall_macro": row["recall_macro"],
            "f1_macro": row["f1_macro"],
            "accuracy": row["accuracy"],
        }
    )

,prompt,avg_confidence,precision_macro,recall_macro,f1_macro,accuracy
0,My investment analyzis tell this company's sec...,0.317271,0.759596,0.727273,0.696465,0.727273
1,I think this company belongs to a sector of:,0.355811,0.760859,0.709091,0.680274,0.709091
2,Ignore legal boilerplate and focus on the econ...,0.276188,0.701515,0.709091,0.680054,0.709091



Top 1 prompt
My investment analyzis tell this company's sector is:
{'avg_confidence': 0.31727118939161303, 'precision_macro': 0.7595959595959595, 'recall_macro': 0.7272727272727273, 'f1_macro': 0.6964646464646465, 'accuracy': 0.7272727272727273}

Top 2 prompt
I think this company belongs to a sector of:
{'avg_confidence': 0.3558106309226291, 'precision_macro': 0.7608585858585858, 'recall_macro': 0.7090909090909091, 'f1_macro': 0.6802737666374029, 'accuracy': 0.7090909090909091}

Top 3 prompt
Ignore legal boilerplate and focus on the economic engine of the firm (what it sells, to whom, and in which market). The closest sector class is:
{'avg_confidence': 0.2761879754422063, 'precision_macro': 0.7015151515151515, 'recall_macro': 0.7090909090909091, 'f1_macro': 0.6800542891451983, 'accuracy': 0.7090909090909091}


In [7]:
reas_estate = sample_df[sample_df["sector"]== "Real Estate"]
re_expl = reas_estate.iloc[0]["raw_text"]

In [8]:
com_serv = sample_df[sample_df["sector"]== "Communication Services"]
com_serv_expl = com_serv.iloc[0]["raw_text"]

In [9]:
example_prompt = top_3_prompts.iloc[0]["prompt"]
example_report = sample_df.iloc[0]["raw_text"]
example_report = com_serv_expl

In [10]:

model_driver.predict_metadata(
    verbolizer=sector_verbolizer,
    prompt=example_prompt,
    text=example_report,
    print_top_tokens=True,
    top_k_tokens=40,
)
#print(example_report[:100])


Top 40 tokens by probability:
Entertainment | 0.165039
entertainment | 0.036865
https        | 0.011597
Technology   | 0.010925
Financial    | 0.010559
Con          | 0.010254
Business     | 0.009644
Enter        | 0.009033
Commun       | 0.008484
Indust       | 0.007263
consumer     | 0.007050
http         | 0.006012
le           | 0.005829
Arts         | 0.005493
Service      | 0.005310
Consumer     | 0.004852
financial    | 0.004547
CON          | 0.004272
Finance      | 0.004272
Event        | 0.003998
Live         | 0.003998
Media        | 0.003891
Hospital     | 0.003891
Industry     | 0.003891
Events       | 0.003540
RE           | 0.003220
Internet     | 0.003128
Invest       | 0.002930
Communications | 0.002762
The          | 0.002365
Tour         | 0.002365
IT           | 0.002289
Sports       | 0.002289
Fin          | 0.002213
Energy       | 0.002213
Food         | 0.002151
Le           | 0.002075
Market       | 0.002075
Global       | 0.001953
Basic        | 0.001892


{'confidence': 0.29901035130023956,
 'probabilities': {'Communication Services': 0.7432342224467009,
  'Financial Services': 0.08381159504431447,
  'Technology': 0.050834697772303684,
  'Consumer Cyclical': 0.04316158803004747,
  'Industrials': 0.033202425266909676,
  'Energy': 0.008976654575881038,
  'Basic Materials': 0.008805334530700686,
  'Utilities': 0.008040181409127618,
  'Consumer Defensive': 0.007620459134232638,
  'Healthcare': 0.007151113723569108,
  'Real Estate': 0.005161728066212687}}